<a href="https://colab.research.google.com/github/kondreddygarivani-bit/project-4/blob/main/W_4Spam_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam.csv


In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

from collections import Counter
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
# Visualization
from wordcloud import WordCloud

# Feature Extraction
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Encoding
from sklearn.preprocessing import LabelEncoder

# Train Test Split
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Sentiment Analysis
from textblob import TextBlob

# Word Embeddings
!pip install gensim
from gensim.models import Word2Vec

# PyTorch
import torch

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.1 MB/s eta 0:00:00


In [ ]:
# Change filename if needed
df = pd.read_csv("spam.csv", encoding='latin-1')

# Rename columns if necessary
df = df.iloc[:, :2]
df.columns = ['Category', 'Message']

print("\nDATASET HEAD\n")
print(df.head())



DATASET HEAD

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
# Change filename if needed
df = pd.read_csv("spam.csv", encoding='latin-1')

# Rename columns if necessary
df = df.iloc[:, :2]
df.columns = ['Category', 'Message']

print("\nDATASET HEAD\n")
print(df.head())



DATASET HEAD

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
#TEXT CLEANING
def clean_text(text):

    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = text.strip()

    return text

df['clean_text'] = df['Message'].apply(clean_text)

print("\nCLEANED TEXT\n")
print(df[['Message', 'clean_text']].head())


CLEANED TEXT

                                             Message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in  a wkly comp to win fa cup final...  
3        u dun say so early hor u c already then say  
4  nah i dont think he goes to usf he lives aroun...  


In [ ]:
nltk.download('punkt_tab')
#TOKENIZATION
df['tokens'] = df['clean_text'].apply(word_tokenize)

print("\nTOKENS\n")
print(df['tokens'].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



TOKENS

0    [go, until, jurong, point, crazy, available, o...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, in, a, wkly, comp, to, win, fa, ...
3    [u, dun, say, so, early, hor, u, c, already, t...
4    [nah, i, dont, think, he, goes, to, usf, he, l...
Name: tokens, dtype: object


In [ ]:
#LEMMATIZATION
lemmatizer = WordNetLemmatizer()

df['lemmatized'] = df['tokens_no_stop'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

print("\nLEMMATIZED WORDS\n")
print(df['lemmatized'].head())



LEMMATIZED WORDS

0    [go, jurong, point, crazy, available, bugis, n...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, wkly, comp, win, fa, cup, final,...
3        [u, dun, say, early, hor, u, c, already, say]
4    [nah, dont, think, go, usf, life, around, though]
Name: lemmatized, dtype: object


In [ ]:
#LABEL ENCODING
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Category'])

print("\nENCODED LABELS\n")
print(df[['Category', 'label']].head())


ENCODED LABELS

  Category  label
0      ham      0
1      ham      0
2     spam      1
3      ham      0
4      ham      0


In [ ]:
#BAG OF WORDS
bow = CountVectorizer()

X_bow = bow.fit_transform(df['clean_text'])

print("\nBAG OF WORDS SHAPE\n")
print(X_bow.shape)


BAG OF WORDS SHAPE

(5572, 8628)


In [ ]:
#TF - IDF
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['clean_text'])

print("\nTF-IDF SHAPE\n")
print(X.shape)


TF-IDF SHAPE

(5572, 8628)


In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, df['label'], test_size=0.2, random_state=42)

#NAIVE BAYES
nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("\nNAIVE BAYES")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))


NAIVE BAYES
Accuracy: 0.9551569506726457


In [ ]:
#WORD FREQUENCY
all_words = []

for words in df['tokens_no_stop']:
    all_words.extend(words)

counter = Counter(all_words)

print("\nTOP 20 WORDS\n")
print(counter.most_common(20))


TOP 20 WORDS

[('u', 1154), ('call', 577), ('im', 466), ('ur', 390), ('get', 390), ('dont', 287), ('go', 285), ('ok', 278), ('free', 278), ('ltgt', 276), ('â£', 271), ('know', 257), ('got', 252), ('like', 244), ('ill', 239), ('good', 236), ('come', 229), ('day', 212), ('time', 208), ('love', 200)]


In [ ]:
#N- GRAMS
ngram_vectorizer = CountVectorizer(ngram_range=(1,2))

X_ngram = ngram_vectorizer.fit_transform(df['clean_text'])

print("\nNGRAM FEATURES\n")
print(ngram_vectorizer.get_feature_names_out()[:20])


NGRAM FEATURES

['aa' 'aa and' 'aah' 'aah bless' 'aah cuddle' 'aah speak' 'aaniye'
 'aaniye pudunga' 'aaooooright' 'aaooooright are' 'aathilove'
 'aathilove lot' 'aathiwhere' 'aathiwhere are' 'ab' 'ab sara' 'abbey'
 'abbey happy' 'abdomen' 'abdomen and']


In [ ]:
#WORD 2 VEC
word2vec_model = Word2Vec(
    sentences=df['tokens_no_stop'],
    vector_size=100,
    window=5,
    min_count=1
)

print("\nWORD2VEC VECTOR FOR 'free'\n")

if 'free' in word2vec_model.wv:
    print(word2vec_model.wv['free'])


WORD2VEC VECTOR FOR 'free'

[-0.25305325  0.37666455  0.23182498  0.1470723   0.04779596 -0.7813591
  0.08536644  1.0215919  -0.44603887 -0.202065   -0.4211312  -0.7037682
 -0.18600765  0.23229618  0.07990727 -0.3129855   0.03050127 -0.60652924
  0.07353096 -0.97709405  0.31078288  0.14034346  0.47004244 -0.2681331
  0.0391402  -0.00780051 -0.40862715 -0.21674615 -0.5570099   0.09677957
  0.50458825  0.15277909  0.26094273 -0.25633386 -0.18130352  0.47602493
  0.03408042 -0.43059954 -0.38099873 -0.9675462   0.03273894 -0.38681352
 -0.21267246  0.05048221  0.42178255 -0.32199308 -0.31535763  0.01875568
  0.4308548   0.340727    0.3374652  -0.3769017  -0.10999461 -0.30061448
 -0.27879772  0.23987252  0.19380844  0.08869344 -0.40022847  0.20308392
  0.06953033  0.15438929 -0.1119646  -0.01505611 -0.5607853   0.37496012
  0.21140462  0.46662518 -0.7193934   0.6608432  -0.42715722  0.16467081
  0.5279738  -0.16184463  0.5724195   0.20477313  0.17797932 -0.10225355
 -0.40164548  0.14132847 

In [ ]:
#PYTORCH
tensor_example = torch.tensor([1,2,3,4,5])

print("\nPYTORCH TENSOR\n")
print(tensor_example)


PYTORCH TENSOR

tensor([1, 2, 3, 4, 5])


In [ ]:
print("\nFINAL DATASET\n")
print(df.head())

print("\nPIPELINE COMPLETED SUCCESSFULLY")


FINAL DATASET

  Category                                            Message  \
0      ham  Go until jurong point, crazy.. Available only ...   
1      ham                      Ok lar... Joking wif u oni...   
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...   
3      ham  U dun say so early hor... U c already then say...   
4      ham  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  \
0  go until jurong point crazy available only in ...   
1                            ok lar joking wif u oni   
2  free entry in  a wkly comp to win fa cup final...   
3        u dun say so early hor u c already then say   
4  nah i dont think he goes to usf he lives aroun...   

                                              tokens  \
0  [go, until, jurong, point, crazy, available, o...   
1                     [ok, lar, joking, wif, u, oni]   
2  [free, entry, in, a, wkly, comp, to, win, fa, ...   
3  [u, dun, say, so, early, hor,